# Track 3: a GRPO run that has headroom

The submission reports two GRPO attempts that moved nothing. Both used **2 generations**, on
policies whose graded reward was already close to saturated: `frac_reward_zero_std` was 0.8–0.95,
so most groups had no advantage to learn from, and the paired test is 0 fixed / 0 broken (p = 1.0).
That is a real result, but it answers "does GRPO help a 96% policy?" and not the question a judge
will ask: **can RL lift this agent at all?**

This kernel starts from a policy that still gets 22/90 hard tasks wrong.

| base | hard-30 | hard-90 | calls |
|---|---:|---:|---:|
| `sft3` (control) | 96.67% (29/30) | **75.56% (68/90)** | 3.27 |
| `sft3` + GRPO, 2 generations (published, flat) | 96.67% | 75.56% | 3.27 |

One variable changes: **4 generations per prompt** instead of 2, on
`data/tasks_train_grpo3.jsonl` (274 rows, easy rows removed), 32 steps, graded + efficiency +
abstain rewards. Same base, same data, same seed as the flat run — so the comparison is
controlled.

| step | time on one T4 |
|---|---:|
| setup, install, 20 offline checks | 5 min |
| merge `sft3` from the Hub | 5 min |
| data + reward preflight (`--check`) | 1 min |
| smoke test, 2 steps | ~10 min |
| GRPO, 32 steps, 4 generations | 1–2 h |
| hard-90, safety, hard-30 | ~2.3 h |

`CUDA_VISIBLE_DEVICES=0` is deliberate: Kaggle's second T4 makes `Trainer` wrap the 3.35B model in
`DataParallel`, which OOMs (seen in the seed kernel).

In [ ]:
import os
from pathlib import Path

from kaggle_secrets import UserSecretsClient

try:
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as exc:
    raise SystemExit(
        "HF_TOKEN is not attached. CohereLabs/tiny-aya-global is gated. Add-ons -> Secrets, "
        "then Save & Run All."
    ) from exc

os.environ["HF_TOKEN"] = token
env = Path("/tmp/hf_env")
env.write_text(f'export HF_TOKEN="{token}"\n', encoding="utf-8")
env.chmod(0o600)
print("HF_TOKEN set (not printed)")

!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /kaggle/working | tail -1

## Setup

`agenttune` is a git dependency, and this is the one place the kernel can fail before doing any
work: the package needs a recent Python. The cell prints the versions so a failure is obvious.

In [ ]:
%%bash
set -euo pipefail
source /tmp/hf_env
cd /kaggle/working
rm -rf lexsi-agentic-hinglish
git clone -q https://github.com/PRINCE2-AI/lexsi-agentic-hinglish.git
cd lexsi-agentic-hinglish
git log --oneline -1
python -V
python -m pip install -q -r requirements.txt
python -c "import agenttune, trl, torch, transformers; print('agenttune', getattr(agenttune, '__version__', '?')); print('trl', trl.__version__, 'torch', torch.__version__, 'transformers', transformers.__version__)"
python scripts/verify_all.py

## Preflight: the data and the reward, no GPU

`--check` walks the first three tasks through the oracle and grades the oracle text with the same
reward function GRPO uses. If the graded reward is not near `MAX_REWARD` here, the reward is broken
and training would be noise.

In [ ]:
%%bash
set -uo pipefail
source /tmp/hf_env
export CUDA_VISIBLE_DEVICES=0
cd /kaggle/working/lexsi-agentic-hinglish
python scripts/train_grpo.py --check --tasks data/tasks_train_grpo3.jsonl

## Merge `sft3` (the control policy) from the Hub

Merging once is cheaper than reloading the adapter for every rollout and every eval.

In [ ]:
%%bash
set -uo pipefail
source /tmp/hf_env
cd /kaggle/working/lexsi-agentic-hinglish
if [ ! -f /kaggle/working/sft3_merged/config.json ]; then
python - <<'PY'
import torch
from huggingface_hub import snapshot_download
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base = "CohereLabs/tiny-aya-global"
repo = snapshot_download("primce/tiny-aya-hinglish-lora", allow_patterns=["sft3_adapter/*"],
                         local_dir="/kaggle/working/sft3_repo")
tokenizer = AutoTokenizer.from_pretrained(base)
model = AutoModelForCausalLM.from_pretrained(base, dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(model, repo, subfolder="sft3_adapter")
merged = model.merge_and_unload()
merged.save_pretrained("/kaggle/working/sft3_merged", safe_serialization=True)
tokenizer.save_pretrained("/kaggle/working/sft3_merged")
print("merged -> /kaggle/working/sft3_merged")
PY
fi
ls -d /kaggle/working/sft3_merged 2>/dev/null || echo "SFT3 MERGE MISSING"

## Smoke test: 2 steps

Catches an OOM or a broken rollout before an hour is spent. The adapter it produces is thrown away.

In [ ]:
%%bash
set -uo pipefail
source /tmp/hf_env
export CUDA_VISIBLE_DEVICES=0
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
cd /kaggle/working/lexsi-agentic-hinglish
time python scripts/train_grpo.py --model /kaggle/working/sft3_merged \
  --tasks data/tasks_train_grpo3.jsonl --max-steps 2 --generations 2 --lr 3e-6 \
  --efficiency --abstain-reward --dtype float16 --t4-profile auto --out runs/grpo_smoke \
  || echo "SMOKE FAILED"
rm -rf runs/grpo_smoke

## The real run: 32 steps, 4 generations

Same base, same task file, same seed as `grpo3` (the published flat run). Group size is the only
change that matters for the advantage, so if this moves hard-90, the difference is attributable.

In [ ]:
%%bash
set -uo pipefail
source /tmp/hf_env
export CUDA_VISIBLE_DEVICES=0
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
cd /kaggle/working/lexsi-agentic-hinglish
time python scripts/train_grpo.py --model /kaggle/working/sft3_merged \
  --tasks data/tasks_train_grpo3.jsonl --max-steps 32 --generations 4 --lr 3e-6 \
  --efficiency --abstain-reward --dtype float16 --t4-profile auto --out runs/grpo4 \
  || echo "GRPO4 FAILED"
ls runs/grpo4/adapter_config.json 2>/dev/null || echo "GRPO4 ADAPTER MISSING"

## hard-90 first: the split with headroom

`hard-30` is saturated at 96.67% for this base, so it cannot show an improvement. hard-90 is where
68/90 leaves room.

In [ ]:
%%bash
set -uo pipefail
source /tmp/hf_env
export CUDA_VISIBLE_DEVICES=0
cd /kaggle/working/lexsi-agentic-hinglish
if [ ! -f runs/grpo4/adapter_config.json ]; then echo "GRPO4 ADAPTER MISSING"; exit 0; fi
time python scripts/run_baseline.py --model /kaggle/working/sft3_merged --adapter runs/grpo4 \
  --tasks data/tasks_hard90_eval.jsonl --no-4bit --out runs/grpo4_hard90 \
  || echo "GRPO4 HARD90 FAILED"
python scripts/job_upload.py --run grpo4_hard90 || echo "UPLOAD FAILED"

## Safety: did RL trade the injection defence away?

This is the regression that matters. `safety_sft3` is the control: 0/24 attacker ledgers, and 1/8
underspecified requests answered with a question.

In [ ]:
%%bash
set -uo pipefail
source /tmp/hf_env
export CUDA_VISIBLE_DEVICES=0
cd /kaggle/working/lexsi-agentic-hinglish
if [ ! -f runs/grpo4/adapter_config.json ]; then echo "GRPO4 ADAPTER MISSING"; exit 0; fi
time python scripts/run_baseline.py --model /kaggle/working/sft3_merged --adapter runs/grpo4 \
  --tasks data/tasks_safety_eval.jsonl --safety --no-4bit --out runs/safety_grpo4 \
  || echo "GRPO4 SAFETY FAILED"
python scripts/job_upload.py --run safety_grpo4 || echo "UPLOAD FAILED"

In [ ]:
%%bash
set -uo pipefail
source /tmp/hf_env
export CUDA_VISIBLE_DEVICES=0
cd /kaggle/working/lexsi-agentic-hinglish
if [ ! -f runs/grpo4/adapter_config.json ]; then echo "GRPO4 ADAPTER MISSING"; exit 0; fi
time python scripts/run_baseline.py --model /kaggle/working/sft3_merged --adapter runs/grpo4 \
  --tasks data/tasks_hard_eval.jsonl --no-4bit --out runs/grpo4_hard30 \
  || echo "GRPO4 HARD30 FAILED"
python scripts/job_upload.py --run grpo4_hard30 --adapter runs/grpo4:grpo4_adapter \
  || echo "UPLOAD FAILED"

## Compare against the control

Pulls `sft3_hard90` from `primce/lexsi-runs` so the paired test has both sides.

In [ ]:
import json
import subprocess
from pathlib import Path

from huggingface_hub import snapshot_download

ROOT = Path("/kaggle/working/lexsi-agentic-hinglish")
snapshot_download("primce/lexsi-runs", repo_type="dataset",
                  allow_patterns=["sft3_hard90/*", "sft3_hard30/*"],
                  local_dir=str(ROOT / "runs"))

KEYS = ("n", "success", "calls", "steps", "ter", "rad", "lcf", "seconds")
for name in ("sft3_hard90", "grpo4_hard90", "safety_grpo4", "sft3_hard30", "grpo4_hard30"):
    path = ROOT / "runs" / name / "summary.json"
    if not path.exists():
        print(f"{name}: MISSING")
        continue
    s = json.loads(path.read_text(encoding="utf-8"))
    row = {k: s.get(k) for k in KEYS if k in s}
    for extra in ("safe", "attack_ledger", "attack_answer", "asked", "invented_expense"):
        if extra in s:
            row[extra] = s[extra]
    row["success_count"] = round((s.get("success") or 0) * (s.get("n") or 0))
    print(name, row)

if (ROOT / "runs" / "grpo4_hard90" / "trajectories.jsonl").exists():
    print(subprocess.run(
        ["python", "scripts/stats.py", "--a", "runs/sft3_hard90", "--b", "runs/grpo4_hard90",
         "--label-a", "sft3", "--label-b", "grpo4"], cwd=ROOT, capture_output=True, text=True,
    ).stdout)

## What to paste back

- `grpo4_hard90.success` and `success_count` vs `sft3_hard90` (0.7556, 68/90).
- the paired McNemar line — fixed/broken counts and p.
- the trainer's `frac_reward_zero_std` from the real run, to say whether 4 generations actually
  produced reward variance.
- `safety_grpo4.attack_ledger` / `attack_answer` / `asked` vs `safety_sft3`.